# Sistema Inteligente de Gestión de Reuniones (RAG + Agente)
### Trabajo Práctico N°2 — Sistemas Inteligentes con RAG o Agentes
**Materia:** Inteligencia Artificial — Ingeniería en Sistemas de Información

**Integrantes:** Juan Pablo Jaca, Nahuel Berli, Gustavo Giampietro, Alexis Mateo

**Caso de negocio:** un asistente que, a partir del audio de una reunión de trabajo,
transcribe lo hablado, genera un resumen (con gráficos/mapa conceptual), permite
consultar en lenguaje natural lo charlado (RAG) y completa automáticamente una
planilla de reunión según Norma ISO 9001 para que el responsable solo la revise.

> Notebook en construcción — las secciones marcadas con `# TODO` quedan pendientes
> de implementación.


## Índice
1. Instalación y configuración
2. Módulo de Transcripción (Speech-to-Text)
3. Módulo de Resumen y Visualización (texto + mapa conceptual)
4. Módulo RAG (indexación y consultas sobre la reunión)
5. Módulo de Completado de Planilla (ISO 9001)
6. Orquestación — Agente principal (Langchain)
7. Casos de prueba
8. Conclusiones y dificultades (defensa oral)


## 1. Instalación y configuración

In [1]:
# TODO: revisar versiones si hay conflictos de dependencias
!pip install -q langchain langchain-community langchain-openai \
    faiss-cpu openai-whisper faster-whisper \
    networkx matplotlib pandas openpyxl python-dotenv pydantic



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# --- Selección de modelo de lenguaje --------------------------------------
# Opción A: GPT vía API de OpenAI
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Opción B: LLaMA local vía Ollama (ya tenés Ollama corriendo localmente)
# from langchain_community.chat_models import ChatOllama
# llm = ChatOllama(model="qwen3.5")  # o "ministral-3:3b"

# TODO: definir cuál se usa finalmente y dejar una sola instancia `llm`
llm = None

# --- Rutas -------------------------------------------------------------
RUTA_AUDIO = "data/reunion_ejemplo.wav"      # TODO: audio de prueba
RUTA_PLANILLA_SALIDA = "output/planilla_reunion.xlsx"


## 2. Módulo de Transcripción (Speech-to-Text)
Convierte el audio de la reunión en texto. Candidato: `whisper` / `faster-whisper`.
Pendiente definir si se agrega diarización (identificar quién habla) — no es
requisito del caso de negocio pero podría enriquecer la planilla (participantes).


In [ ]:
def transcribir_audio(ruta_audio: str) -> str:
    """
    Transcribe un archivo de audio a texto plano.

    TODO:
      - Cargar modelo whisper (tamaño a definir: base/small/medium según recursos)
      - Ejecutar transcripción
      - (Opcional) diarización de hablantes
      - Devolver texto plano de la transcripción
    """
    transcripcion = ""  # TODO: implementar
    return transcripcion


## 3. Módulo de Resumen y Visualización
Genera un resumen ejecutivo de la reunión y, si aporta valor, un mapa conceptual
o gráfico de los temas tratados.


In [ ]:
def generar_resumen(transcripcion: str) -> str:
    """
    Genera un resumen de la transcripción usando una chain de Langchain
    (map_reduce o refine, según longitud del texto).

    TODO:
      - Splitear transcripción en chunks (RecursiveCharacterTextSplitter)
      - Armar chain de resumen (load_summarize_chain)
      - Ejecutar y devolver el resumen
    """
    resumen = ""  # TODO: implementar
    return resumen


def graficar_mapa_conceptual(resumen: str):
    """
    Arma un grafo simple (networkx) con los temas/relaciones principales
    detectados en el resumen.

    TODO:
      - Extraer temas/entidades clave (vía LLM o keyword extraction)
      - Construir grafo de relaciones
      - Dibujar con networkx + matplotlib
    """
    pass  # TODO: implementar


## 4. Módulo RAG — Indexación y Consultas
Permite responder preguntas sobre lo charlado en la reunión (ej. *"¿qué
problemáticas se discutieron y con qué prioridad?"*) recuperando los fragmentos
relevantes de la transcripción.


In [ ]:
def crear_vectorstore(transcripcion: str):
    """
    Indexa la transcripción para recuperación semántica.

    TODO:
      - Splitear transcripción en chunks
      - Generar embeddings (OpenAIEmbeddings / HuggingFaceEmbeddings)
      - Guardar en FAISS (o Chroma)
      - Devolver el vectorstore
    """
    vectorstore = None  # TODO: implementar
    return vectorstore


def consultar_reunion(pregunta: str, vectorstore) -> str:
    """
    Responde una pregunta en base a lo charlado en la reunión (RetrievalQA).

    Ejemplo de uso pensado por el equipo:
      consultar_reunion("¿qué problemáticas se discutieron y en qué orden
                          de prioridad?", vectorstore)
      -> debería devolver un listado de problemáticas con su criticidad,
         rescatado de los fragmentos más relevantes de la transcripción.

    TODO:
      - Armar retriever a partir del vectorstore
      - Armar RetrievalQA / chain con el `llm`
      - Ejecutar consulta y devolver respuesta
    """
    respuesta = ""  # TODO: implementar
    return respuesta


## 5. Módulo de Completado de Planilla (Norma ISO 9001)
A partir de la transcripción y el resumen, completa automáticamente la planilla
estandarizada de reunión (para que el responsable solo la revise y ajuste).


In [ ]:
from pydantic import BaseModel, Field
from typing import List

class PlanillaReunionISO9001(BaseModel):
    """Estructura de la planilla estandarizada de reunión (ISO 9001)."""
    fecha: str = Field(description="Fecha de la reunión")
    participantes: List[str] = Field(description="Participantes identificados")
    temas_tratados: List[str] = Field(description="Temas discutidos")
    acuerdos: List[str] = Field(description="Acuerdos/decisiones tomadas")
    responsables: List[str] = Field(description="Responsables por cada acuerdo")
    proximos_pasos: List[str] = Field(description="Próximos pasos / seguimiento")
    # TODO: completar campos según la planilla real usada en la fábrica
    #       (relevar con el contacto de la empresa si hace falta más detalle)


In [ ]:
def completar_planilla(transcripcion: str, resumen: str) -> PlanillaReunionISO9001:
    """
    Completa la planilla de reunión usando salida estructurada del LLM
    (with_structured_output sobre PlanillaReunionISO9001).

    TODO:
      - Armar prompt con transcripción + resumen
      - Invocar llm.with_structured_output(PlanillaReunionISO9001)
      - Devolver la planilla completada
    """
    planilla = None  # TODO: implementar
    return planilla


def exportar_planilla(planilla: PlanillaReunionISO9001, ruta_salida: str):
    """
    Exporta la planilla completada a Excel (formato que el responsable
    pueda revisar/editar directamente).

    TODO:
      - Convertir planilla a DataFrame
      - Guardar como .xlsx en `ruta_salida`
    """
    pass  # TODO: implementar


## 6. Orquestación — Agente principal (Langchain)
Combina los módulos anteriores como *tools* de un agente, o como un pipeline
secuencial simple (a definir cuál conviene según lo visto en clase).


In [ ]:
# TODO: decidir enfoque final -> Agente con tools (create_react_agent / AgentExecutor)
#       o pipeline secuencial simple (transcribir -> resumir -> indexar -> planilla)
#
# Boceto de tools si se opta por agente:
#
# from langchain.agents import AgentExecutor, create_react_agent
# from langchain.tools import Tool
#
# tools = [
#     Tool(name="ConsultarReunion", func=..., description="Responde preguntas sobre lo charlado"),
#     Tool(name="CompletarPlanilla", func=..., description="Completa la planilla ISO 9001"),
# ]
# TODO: implementar


## 7. Casos de prueba

In [ ]:
# TODO: correr el pipeline completo sobre un audio de prueba y validar salida
#
# transcripcion = transcribir_audio(RUTA_AUDIO)
# resumen = generar_resumen(transcripcion)
# vectorstore = crear_vectorstore(transcripcion)
# print(consultar_reunion("¿qué problemáticas se discutieron y con qué prioridad?", vectorstore))
# planilla = completar_planilla(transcripcion, resumen)
# exportar_planilla(planilla, RUTA_PLANILLA_SALIDA)


## 8. Conclusiones y dificultades
*(Para completar de cara a la defensa oral: decisiones de diseño tomadas,
dificultades encontradas y cómo se resolvieron, resultados obtenidos.)*

- Caso de negocio: _pendiente_
- Decisiones de diseño: _pendiente_
- Dificultades: _pendiente_
- Resultados: _pendiente_
